# Tarea 2 – Extracción OCR (Corpus A: documentos)
- Fecha de extracción: 25/10/2026
- PaddleOCR 3.7.0, Python 3.11.17 (entorno conda `tarea2`)
- Alcance: solo Corpus A (documentos). El Corpus B (audio) no se incluyó en esta entrega.
- Asistente de IA usado: Claude (chat), capturas en `evidencias_ia/`
- Fuentes y licencias: ver `datos/fuentes.csv`

In [1]:
import os, hashlib, pandas as pd

carpeta = "datos/raw/documentos"
archivos = sorted(f for f in os.listdir(carpeta) if f.lower().endswith(".jpg"))
print(len(archivos), "archivos")

def tipo(n):
    return "impreso" if n <= 8 else ("foto" if n <= 16 else "manuscrito")

filas, hashes = [], {}
for f in archivos:
    n = int(f[4:7])
    ruta = f"{carpeta}/{f}"
    h = hashlib.sha256(open(ruta, "rb").read()).hexdigest()
    hashes.setdefault(h, []).append(f)
    filas.append({"id": f[:-4], "archivo": ruta, "tipo": tipo(n), "fuente": "",
                  "texto_ocr": "", "modelo_ocr": "", "texto_referencia": ""})

df = pd.DataFrame(filas)
print("Duplicados:", [v for v in hashes.values() if len(v) > 1])
print("MB mayores a 3:", [f for f in archivos if os.path.getsize(f"{carpeta}/{f}") > 3e6])
os.makedirs("datos", exist_ok=True)
df.to_csv("datos/documentos.csv", index=False)
pd.DataFrame({"id": df["id"], "modalidad": "documento", "fuente_url": "",
              "licencia": "", "acceso": ""}).to_csv("datos/fuentes.csv", index=False)
df.head()

25 archivos
Duplicados: []
MB mayores a 3: []


,id,archivo,tipo,fuente,texto_ocr,modelo_ocr,texto_referencia
0,doc_001,datos/raw/documentos/doc_001.jpg,impreso,,,,
1,doc_002,datos/raw/documentos/doc_002.jpg,impreso,,,,
2,doc_003,datos/raw/documentos/doc_003.jpg,impreso,,,,
3,doc_004,datos/raw/documentos/doc_004.jpg,impreso,,,,
4,doc_005,datos/raw/documentos/doc_005.jpg,impreso,,,,


In [1]:
import time
from paddleocr import PaddleOCR

ocr = PaddleOCR(lang="es",
                use_doc_orientation_classify=False,
                use_doc_unwarping=False,
                use_textline_orientation=False,
                enable_mkldnn=False)
t = time.time()
res = ocr.predict("datos/raw/documentos/doc_001.jpg")
print(round(time.time() - t, 1), "s")
print(res[0]["rec_texts"][:10])

d:\Python2026\envs\tarea2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Python2026\envs\tarea2\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\acer\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\acer\.paddlex\official_models\PP-OCRv6_medium_re

104.2 s
['1', 'CONSTITUCIÓN POLÍTICA DEL PERÚ', 'PREÁM BULO', 'EL CONGRESO CONSTITUYENTE DEMOCRÁTICO, INVOCANDO A DIOS', 'TODOPODEROSO, OBEDECIENDO EL MANDATO DEL PUEBLO PERUANO Y', 'RECORDANDO EL SACRIFICIO DE TODAS LAS GENERACIONES QUE NOS', 'HAN PRECEDIDO EN NUESTRA PATRIA, HA RESUELTO DAR LA SIGUIENTE', 'CONSTITUCION:', 'TÍTULOI', 'DE LA PERSONA Y DE LA SOCIEDAD']


In [2]:
import os, time, pandas as pd
from paddleocr import PaddleOCR

ocr = PaddleOCR(lang="es", use_doc_orientation_classify=False,
                use_doc_unwarping=False, use_textline_orientation=False,
                enable_mkldnn=False)

carpeta = "datos/raw/documentos"
filas = []
for f in sorted(os.listdir(carpeta)):
    if not f.lower().endswith(".jpg"):
        continue
    n = int(f[4:7])
    t = time.time()
    r = ocr.predict(f"{carpeta}/{f}")[0]
    texto = " ".join(r["rec_texts"])
    filas.append({"id": f[:-4], "archivo": f"{carpeta}/{f}",
                  "tipo": "impreso" if n <= 8 else ("foto" if n <= 16 else "manuscrito"),
                  "fuente": "", "texto_ocr": texto,
                  "modelo_ocr": "PaddleOCR 3.7.0 (lang=es)",
                  "texto_referencia": "", "segundos": round(time.time() - t, 1)})
    print(f, len(texto), "caracteres", round(time.time() - t, 1), "s")

df = pd.DataFrame(filas)
os.makedirs("datos", exist_ok=True)
df.drop(columns="segundos").to_csv("datos/documentos.csv", index=False)
df.to_csv("datos/raw/documentos/ocr_con_tiempos.csv", index=False)
print(df.shape)
df.head()

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\acer\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\acer\.paddlex\official_models\PP-OCRv6_medium_rec`.


doc_001.jpg 2070 caracteres 103.0 s
doc_002.jpg 2492 caracteres 120.5 s
doc_003.jpg 2200 caracteres 104.8 s
doc_004.jpg 1874 caracteres 99.3 s
doc_005.jpg 2575 caracteres 124.4 s
doc_006.jpg 3384 caracteres 148.5 s
doc_007.jpg 3712 caracteres 177.2 s
doc_008.jpg 6393 caracteres 202.0 s
doc_009.jpg 3269 caracteres 71.0 s
doc_010.jpg 545 caracteres 131.7 s
doc_011.jpg 1138 caracteres 69.6 s
doc_012.jpg 1740 caracteres 77.7 s
doc_013.jpg 1857 caracteres 81.0 s
doc_014.jpg 490 caracteres 43.9 s
doc_015.jpg 672 caracteres 42.2 s
doc_016.jpg 848 caracteres 51.3 s
doc_017.jpg 2498 caracteres 171.0 s
doc_018.jpg 1103 caracteres 114.9 s
doc_019.jpg 983 caracteres 113.3 s
doc_020.jpg 1127 caracteres 112.6 s
doc_021.jpg 2306 caracteres 304.8 s
doc_022.jpg 1931 caracteres 139.8 s
doc_023.jpg 559 caracteres 106.0 s
doc_024.jpg 2548 caracteres 150.9 s
doc_025.jpg 2415 caracteres 156.7 s
(25, 8)


,id,archivo,tipo,fuente,texto_ocr,modelo_ocr,texto_referencia,segundos
0,doc_001,datos/raw/documentos/doc_001.jpg,impreso,,1 CONSTITUCIÓN POLÍTICA DEL PERÚ PREÁM BULO EL...,PaddleOCR 3.7.0 (lang=es),,103.0
1,doc_002,datos/raw/documentos/doc_002.jpg,impreso,,2 El secreto bancario y la reserva tributaria ...,PaddleOCR 3.7.0 (lang=es),,120.5
2,doc_003,datos/raw/documentos/doc_003.jpg,impreso,,PUNTO CENTRO DE UNIVERSIDAD INVESTIGACIÓN DEL ...,PaddleOCR 3.7.0 (lang=es),,104.8
3,doc_004,datos/raw/documentos/doc_004.jpg,impreso,,causales de juicio político (impeachment) cont...,PaddleOCR 3.7.0 (lang=es),,99.3
4,doc_005,datos/raw/documentos/doc_005.jpg,impreso,,"donde concentración, poder de mercado y eficie...",PaddleOCR 3.7.0 (lang=es),,124.4
